### Check number of images for each mushroom

In [4]:
import os

parent_directory = "./data/archive/data/data"

below = []

for item in os.listdir(parent_directory):
    item_path = os.path.join(parent_directory, item)
    
    if os.path.isdir(item_path):
        file_count = len([
            f for f in os.listdir(item_path) 
            if os.path.isfile(os.path.join(item_path, f))
        ])
        print(f"{item} | {file_count}")
        if file_count < 10:
            below.append((item, file_count))

for folder, count in below:
    print(f"UH OH {folder} only has: {count} files")

almond_mushroom | 12
amanita_gemmata | 15
amethyst_chanterelle | 15
amethyst_deceiver | 15
aniseed_funnel | 15
ascot_hat | 15
bay_bolete | 15
bearded_milkcap | 15
beechwood_sickener | 15
beefsteak_fungus | 15
birch_polypore | 15
birch_woodwart | 12
bitter_beech_bolete | 15
bitter_bolete | 15
blackening_brittlegill | 15
blackening_polypore | 15
blackening_waxcap | 15
black_bulgar | 15
black_morel | 15
blue_roundhead | 15
blushing_bracket | 15
blushing_rosette | 11
blushing_wood_mushroom | 15
bovine_bolete | 15
bronze_bolete | 15
brown_birch_bolete | 15
brown_rollrim | 15
bruising_webcap | 15
butter_cap | 15
cauliflower_fungus | 15
cedarwood_waxcap | 10
chanterelle | 15
charcoal_burner | 15
chestnut_bolete | 15
chicken_of_the_woods | 15
cinnamon_bracket | 9
clouded_agaric | 15
clustered_domecap | 12
common_bonnet | 15
common_inkcap | 15
common_morel | 15
common_puffball | 15
common_rustgill | 15
crimped_gill | 15
crimson_waxcap | 15
cucumber_cap | 15
curry_milkcap | 15
deadly_fibrecap | 

### Check bit depth / possible colors

In [ ]:
from PIL import Image

img = Image.open("data/archive/data/data/almond_mushroom/0.png")

mode_info = {
    '1': (1, 2),                  # 1-bit, 2 colors (black and white)
    'L': (8, 256),                # 8-bit, 256 shades of grayscale
    'P': (8, 256),                # 8-bit, palette-based (up to 256 colors)
    'RGB': (24, 16777216),        # 24-bit true color (8 bits x 3 channels)
    'RGBA': (32, 4294967296),     # 32-bit true color with transparency (8 bits x 4 channels)
    'I': (32, "32-bit integer"),
    'F': (32, "32-bit float")
}

bits, possible_colors = mode_info.get(img.mode, (None, "Unknown"))

print(f"Image Mode: {img.mode}")
print(f"Bit Depth: {bits} bits")
print(f"Total Possible Colors: {possible_colors}")

Image Mode: RGBA
Bit Depth: 32 bits
Total Possible Colors: 4294967296


### Build dataset for training

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

transform = transforms.Compose([
    # transforms.Resize((512, 512)),
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root='data/archive/data/data', transform=transform)

targets = dataset.targets

train_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=targets,
    random_state=42
)

train_dataset = Subset(dataset, train_indices)
test_dataset = Subset(dataset, test_indices)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Testing images: {len(test_dataset)}")

Training images: 2497
Testing images: 625


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = resnet18(weights=ResNet18_Weights.DEFAULT)

num_classes = len(dataset.classes) 
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

Epoch [1/10], Loss: 5.8086
Epoch [2/10], Loss: 5.5112
Epoch [3/10], Loss: 5.4147
Epoch [4/10], Loss: 5.3651
Epoch [5/10], Loss: 5.3261
Epoch [6/10], Loss: 5.3096
Epoch [7/10], Loss: 5.2913
Epoch [8/10], Loss: 5.2730
Epoch [9/10], Loss: 5.2556
Epoch [10/10], Loss: 5.2431


In [8]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 1.12%
